In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
import healpix_geo
import numpy as np
import xarray as xr
import xdggs  # noqa: F401
import zarr

import hcc  # noqa: F401

In [ ]:
level = 17
cell_ids = np.arange(0 * 4**level, 0 * 4**level + 4**11, dtype="uint64")
lon, lat = healpix_geo.nested.healpix_to_lonlat(cell_ids, level, ellipsoid="sphere")
data = np.abs(np.cos(np.deg2rad(90 * lat)) * np.sin(np.deg2rad(180 * lon)))

ds = xr.Dataset(
    {"data": ("cells", data)}, coords={"cell_ids": ("cells", cell_ids)}
).dggs.decode(
    {
        "grid_name": "healpix",
        "level": 17,
        "indexing_scheme": "nested",
        "ellipsoid": "sphere",
    }
)

In [ ]:
ds.dggs.explore()

In [ ]:
compressed_store = zarr.storage.MemoryStore()
encoding = {
    "data": {
        "serializer": {"name": "hcc", "configuration": {"delta": 2, "top_k": 100}},
        "chunks": 4**9,
    }
}
ds.dggs.encode(convention="zarr").to_zarr(
    compressed_store,
    mode="w",
    encoding=encoding,
    consolidated=False,
)

In [ ]:
uncompressed_store = zarr.storage.MemoryStore()
ds.dggs.encode(convention="zarr").to_zarr(
    uncompressed_store, mode="w", consolidated=False
)

In [ ]:
def store_size(store):
    sizes = {
        key: value.as_array_like().nbytes for key, value in store._store_dict.items()
    }

    return sum(sizes.values())

In [ ]:
store_size(uncompressed_store) / 10**6

In [ ]:
store_size(compressed_store) / 10**6

In [ ]:
print(compressed_store._store_dict["data/zarr.json"].to_bytes().decode())

In [ ]:
reopened = (
    xr.open_dataset(compressed_store, engine="zarr", consolidated=False)
    .dggs.decode(convention="zarr")
    .load()
)
reopened

In [ ]:
reopened.dggs.explore()

In [ ]:
(ds - reopened).dggs.explore()